# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² mass spectrometry dataset using the `mlcroissant` library. Each record set, field, and column is referenced by its unique `@id`, per FAIR and Croissant best practices.

### Dataset Source

The dataset is described by a Croissant schema available at:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure mlcroissant and pandas are installed
!pip install mlcroissant pandas

## 1. Data Loading

First, load dataset metadata and list available record sets.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata fields with dot notation
meta = dataset.metadata
print(f"Dataset Name: {meta.name}\n\nDescription: {meta.description}")
print(f"Published: {meta.datePublished} | Version: {meta.version}")
print(f"Keywords: {', '.join(meta.keywords)}")


## 2. Data Overview

Explore available record sets, fields, and their `@id`s.

The Croissant dataset schema describes the structure as follows. We will list all `RecordSet` objects, and for each record set, show its `@id` and contained fields and columns referenced by their `@id` fields.

In [ ]:
# List all RecordSets and their field/column @ids
record_sets = []

for rs in dataset.metadata.recordSet:
    print(f"RecordSet Name: {rs.name}, @id: {rs['@id']}")
    record_sets.append(rs['@id'])
    # List fields
    if hasattr(rs, 'field'):
        print("  Fields:")
        for f in rs.field:
            print(f"    - {f.get('@id', '')}: {f.get('name', '')}")
    # List columns
    if hasattr(rs, 'column'):
        print("  Columns:")
        for col in rs.column:
            print(f"    - {col.get('@id', '')}: {col.get('name', '')}")
    print()
if not record_sets:
    print("No RecordSets detected in the metadata.\nIt is possible that all records are part of a default or single main RecordSet. Let's use the dataset interface to enumerate available records.")


## 3. Data Extraction

Load records from all available record sets into Pandas DataFrames for analysis. All references use RecordSet `@id` values as variables for reproducible downstream processing.


In [ ]:
# Try to infer main RecordSet if none found above
if not record_sets:
    # common mlcroissant behavior: dataset may have a single default RecordSet
    print('Trying to list records using default RecordSet')
    try:
        records = list(dataset.records())
        print(f"Records loaded: {len(records)}")
        main_df = pd.DataFrame(records)
        print("Available columns:", main_df.columns.tolist())
    except Exception as e:
        print("No records found or unable to read records.", e)
    # Save for later use
    dataframes = {'main': main_df}
    main_record_set_id = 'main'
else:
    dataframes = {}
    for record_set_id in record_sets:
        print(f"Extracting records from RecordSet: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        print("Columns:", dataframes[record_set_id].columns.tolist())
    main_record_set_id = record_sets[0] if record_sets else None


## 4. Exploratory Data Analysis (EDA)

Now, select a numeric field by its `@id` or column name, filter on its value, normalize it, and optionally group by another field. All field/column references are made by their unique `@id` or column name.

In [ ]:
# Replace with actual numeric field @id or column name after inspecting above
# For demonstration, we try common column names from proteomics (e.g., 'MW_kDa', 'Coverage_Percent', 'Peptide_Count').
import numpy as np
df = dataframes[main_record_set_id]

candidate_numeric_fields = ['MW_kDa', 'Coverage_Percent', 'Peptide_Count', 'Abundance']
numeric_field = None
for cand in candidate_numeric_fields:
    if cand in df.columns:
        numeric_field = cand
        break
if numeric_field is None:
    print("No common numeric field found, listing available numeric fields...")
    # Guess by dtype
    for col in df.select_dtypes(include=[np.number]).columns:
        print(col)
    numeric_field = df.select_dtypes(include=[np.number]).columns[0] if len(df.select_dtypes(include=[np.number]).columns) > 0 else df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Filter criterion (demonstration: choose a reasonable threshold)
threshold = df[numeric_field].mean() if numeric_field in df.columns else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered to records with {numeric_field} > {threshold:.2f}, count: {len(filtered_df)}")

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
candidate_group_fields = ['Protein_Description', 'Description', 'Sample_Type']
group_field = None
for cand in candidate_group_fields:
    if cand in df.columns:
        group_field = cand
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())


## 5. Visualization

Visualize the distribution of the selected numeric field and show relationships with any group field found.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# If a group_field exists and is not too high cardinality, show boxplot
if group_field and df[group_field].nunique() < 12:
    plt.figure(figsize=(12,6))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()


## 6. Conclusion

In this notebook, we loaded and analyzed the FAIR² mass spectrometry dataset using `mlcroissant`, referencing all entities by their unique `@id` or column names. We performed basic EDA: loaded the schema, inspected available fields, normalized numeric values, and visualized their distributions. For advanced applications, explore all columns and their `@id`s using the schema browser, and adapt the analysis to specific protein biomarkers or experimental groupings as needed.